# 01 — Environment Check

Kiểm tra môi trường, API keys, và kết nối PostgreSQL trước khi chạy pipeline.

In [1]:
# ── Cài dependencies ──────────────────────────────────────────────────────
%pip install -r ../requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ── Project root setup ────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f'[OK]  Project root: {PROJECT_ROOT}')

[OK]  Project root: P:\study_again\gold_time_prediction


In [ ]:
# ── Kiểm tra API Keys ─────────────────────────────────────────────────────
from src.utils.config_loader import (
    FRED_KEY_VALID, EIA_KEY_VALID,
    DATA_START_DATE, DATA_END_DATE,
    FREEGOLD_PATH, YFINANCE_PATH, FRED_PATH, EIA_PATH,
    PG_SCHEMA_RAW, PG_SCHEMA_STAGING, PG_SCHEMA_FEATURES,
    SQL_DIR,
)

print(f'[INFO] Data range     : {DATA_START_DATE} → {DATA_END_DATE or "today"}')
print(f'[INFO] FRED API key   : {"✓ valid" if FRED_KEY_VALID else "✗ MISSING — set FRED_API_KEY in .env"}')
print(f'[INFO] EIA API key    : {"✓ valid" if EIA_KEY_VALID else "⚠ missing — sẽ dùng yfinance fallback"}')
print(f'[INFO] PG Schemas     : {PG_SCHEMA_RAW}, {PG_SCHEMA_STAGING}, {PG_SCHEMA_FEATURES}')
print(f'[INFO] SQL Dir        : {SQL_DIR}')

[INFO] Data range     : 2000-01-01 → today
[INFO] FRED API key   : ✓ valid
[INFO] EIA API key    : ✓ valid
[INFO] PG Schemas     : raw, staging, features
[INFO] SQL Dir        : P:\study_again\gold_time_prediction\sql


In [ ]:
# ── Kiểm tra kết nối PostgreSQL ───────────────────────────────────────────
from src.data.storage.postgres_client import get_engine, table_exists, get_row_count

try:
    engine = get_engine()
    with engine.connect() as conn:
        from sqlalchemy import text
        result = conn.execute(text('SELECT version()')).scalar()
    print(f'[OK]  PostgreSQL connected')
    print(f'      Version: {result[:60]}...')
except Exception as e:
    print(f'[ERR] PostgreSQL connection FAILED: {e}')
    print('      Kiểm tra DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME trong .env')

2026-05-04 23:16:45 | INFO     | src.data.storage.postgres_client | PG engine created
[OK]  PostgreSQL connected
      Version: PostgreSQL 16.13, compiled by Visual C++ build 1944, 64-bit...


In [ ]:
# ── Kiểm tra trạng thái Database ─────────────────────────────────────────
import pandas as pd

tables_to_check = [
    ('raw',      'gold_prices'),
    ('raw',      'yfinance_daily'),
    ('raw',      'fred_daily'),
    ('raw',      'fred_monthly'),
    ('raw',      'eia_oil'),
    ('staging',  'daily_master'),
    ('features', 'price_indicators'),
    ('features', 'momentum_indicators'),
    ('features', 'trend_indicators'),
    ('features', 'macro_features'),
    ('features', 'ratio_features'),
    ('features', 'sliding_windows'),
    ('features', 'target_labels'),
    ('features', 'master_features'),
]

rows = []
for schema, table in tables_to_check:
    exists = table_exists(schema, table)
    count  = get_row_count(schema, table) if exists else 0
    status = '✓' if exists and count > 0 else ('⚠ EMPTY' if exists else '✗ NOT FOUND')
    rows.append({'schema.table': f'{schema}.{table}', 'rows': count, 'status': status})

df_status = pd.DataFrame(rows)
print(df_status.to_string(index=False))

print('\n[NEXT] Chạy notebooks/02_ingestion_api.ipynb để populate database')

2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:46 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:47 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:47 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:47 | INFO     | src.data.storage.postgres_client | PG engine created
2026-05-04 23:16:47 | INFO     | src.data.storage.post

In [ ]:
# ── Tạo thư mục cần thiết ─────────────────────────────────────────────────
from src.utils.config_loader import FREEGOLD_PATH, YFINANCE_PATH, FRED_PATH, EIA_PATH

dirs = [FREEGOLD_PATH, YFINANCE_PATH, FRED_PATH, EIA_PATH]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f'[OK]  Dir ready: {d.relative_to(PROJECT_ROOT)}')

print('\n[DONE] Environment check hoàn tất!')

[OK]  Dir ready: data\raw\incoming\freegoldapi
[OK]  Dir ready: data\raw\incoming\yfinance
[OK]  Dir ready: data\raw\incoming\fred
[OK]  Dir ready: data\raw\incoming\eia

[DONE] Environment check hoàn tất!
